<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>토크나이저 기본(Tokenizer Fundamentals)</strong>에 대해 학습합니다.</br></br>
>BPE 알고리즘의 원리를 이해하고 서브워드 토크나이징을 직접 학습해봅시다.

</br>

# 토크나이저 (Tokenizer)
> 토크나이저는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">텍스트를 모델이 처리할 수 있는 토큰 단위로 분리</mark>하는 전처리 도구입니다.

컴퓨터는 텍스트를 직접 이해하지 못합니다. 신경망은 행렬 연산만 수행하므로, 모든 입력은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">숫자(정수 또는 실수)</mark>여야 합니다. 텍스트를 숫자로 변환하는 과정이 바로 토크나이징이며, 이것이 NLP 전처리의 첫 단계입니다. 텍스트는 컴퓨터 내부에서 유니코드 숫자의 나열로 저장되고, 모델이 알고 있는 토큰의 집합인 어휘 사전(Vocabulary)은 정수 인덱스로 관리됩니다.</br>

그렇다면 텍스트를 어떤 단위로 쪼갤 것인가가 핵심 설계 결정입니다. <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">단어 단위</mark>로 분리하면 "고양이가"와 "고양이는"을 다른 단어로 취급하여 어휘가 폭발적으로 커지고 미등록 단어(OOV)가 발생합니다. <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">문자 단위</mark>로 처리하면 OOV는 없지만 시퀀스가 지나치게 길어집니다. <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">서브워드 단위</mark>는 자주 나오는 조합은 하나의 토큰으로, 희귀한 단어는 쪼개서 처리하여 두 방식의 균형을 잡습니다. 현대 언어 모델(GPT, BERT, LLaMA)이 모두 서브워드 방식을 채택한 이유는 **어휘 크기와 표현력 사이의 최적 균형** 때문이며, 토크나이저의 설계가 바뀌면 같은 모델도 성능이 달라지므로 NLP 파이프라인에서 가장 근본적인 선택입니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">방식</th>
      <th style="text-align:center">단위</th>
      <th>예시</th>
      <th>문제점</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">단어 기반</td><td style="text-align:center">단어</td><td><code>["안녕", "하세요"]</code></td><td>OOV (미등록 단어)</td></tr>
    <tr><td style="text-align:center">문자 기반</td><td style="text-align:center">글자</td><td><code>["안", "녕", "하"]</code></td><td>시퀀스 길이 폭증</td></tr>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">서브워드 기반</mark></td><td style="text-align:center">서브워드</td><td><code>["안녕", "하", "세요"]</code></td><td>균형 잡힌 접근</td></tr>
  </tbody>
</table>

</br>

# BPE (Byte Pair Encoding)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">가장 빈번한 문자 쌍을 반복적으로 병합</mark>하여 서브워드 사전을 구축하는 알고리즘입니다.

</br>

## get_stats 구현
> 현재 토큰 시퀀스에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">인접 토큰 쌍의 빈도</mark>를 수집합니다.

In [ ]:
# TODO 1: vocab 딕셔너리의 각 단어에서 인접 심볼 쌍의 빈도를 집계하여 반환하는 함수를 정의하고, 테스트 vocab으로 상위 3개 쌍을 출력해봅시다.

def get_stats(vocab):
    pairs = {}
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i+1])
            pairs[pair] = pairs.get(pair, 0) + freq
    return pairs

# 테스트
vocab = {"l o w </w>": 5, "l o w e r </w>": 2, "n e w </w>": 6}
stats = get_stats(vocab)
print("빈도 상위 쌍:")
for pair, freq in sorted(stats.items(), key=lambda x: -x[1])[:3]:
    print(f"  {pair}: {freq}")

</br>

## BPE 전체 흐름

In [ ]:
# TODO 2: 정규식 모듈을 활용하여, 주어진 쌍을 vocab에서 병합하는 함수를 정의하고, 빈도 통계와 병합을 5회 반복 호출하여 BPE 병합 과정을 출력해봅시다.

import re

def merge_vocab(pair, vocab):
    new_vocab = {}
    bigram = re.escape(' '.join(pair))
    pattern = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in vocab:
        new_word = pattern.sub(''.join(pair), word)
        new_vocab[new_word] = vocab[word]
    return new_vocab

vocab = {"l o w </w>": 5, "l o w e r </w>": 2, "n e w </w>": 6}

for i in range(5):
    pairs = get_stats(vocab)
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    print(f"Merge {i+1}: {best[0]} + {best[1]} → {''.join(best)}  (freq={pairs[best]})")

print(f"\n최종 사전: {vocab}")

💡BPE의 장점
> 미등록 단어(OOV)를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">서브워드 조합으로 표현</mark>할 수 있습니다.</br>
> GPT, RoBERTa 등 대부분의 현대 LLM이 BPE 변형을 사용합니다.

💡사전 크기는 어떻게 정할까?
> 일반적으로 30,000~50,000 토큰이 많이 사용됩니다.</br>
> 사전이 크면 시퀀스가 짧아지고, 작으면 서브워드 분할이 세밀해집니다.